<a href="https://colab.research.google.com/github/bhaibachaopls-web/Flyrank_repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhaibachaopls-web/Flyrani_repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

#### My lane : Classification.
I am predicting a discrete, categorical outcome: whether a piece of content will experience significant traffic decay or remain stable/grow. I'm not clustering unlabeled data or sort items by relevance or predicting a continuous numerical value like exact future page views

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_decaying'] = np.where(df['trend_pct'] < -20, 1, 0)

df

target_classes = df['is_decaying'].unique()
target_classes.sort()

print(f"Target variable name: 'is_decaying'")
print(f"Discrete classes found: {target_classes}")
print(f"Number of classes: {len(target_classes)} (Binary Classification)")
print(f"Target data type: {df['is_decaying'].dtype}")

Target variable name: 'is_decaying'
Discrete classes found: [0 1]
Number of classes: 2 (Binary Classification)
Target data type: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

##### Target : is_decaying (binary : 1)
This label is a proxy created via a defined mathematical rule applied to an observed metric. The dataset provides the raw observed outcome as a continuous percentage change in traffic (trend_pct). Because this is a classification task, I cannot predict that raw number directly. Instead, I am applying a strict rule: if a page's trend_pct drops below -20%, it is assigned the class of 1 (decaying).

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

raw_metric = df['trend_pct']
df['is_decaying'] = np.where(raw_metric < -20, 1, 0)

print(df[['content_id', 'trend_pct', 'is_decaying']].head(5))


             content_id  trend_pct  is_decaying
0  content_304f48230142      -41.4            1
1  content_a1fb4e703a9e      -57.7            1
2  content_9aa793d4d895      -60.9            1
3  content_331d6c4de07b      -13.8            0
4  content_d99b7a2d90ca      -34.7            1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Metric: Precision (for Class 1 - Decaying)

Because this model triggers a manual, human-in-the-loop action, the primary constraint is the editorial team's bandwidth. If the model flags a page for an update, the team will spend hours rewriting it. High precision minimizes false alarms and ensures we do not waste expensive editorial budget on healthy pages.

A good precision number is >=75%.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

base_rate_precision = df['is_decaying'].mean()

target_precision = 0.75

print(f"Baseline Precision (Random/Always Predict 1): {base_rate_precision * 100:.1f}%")
print(f"Target 'Good' Precision for Model: >= {target_precision * 100:.1f}%")
print(f"Required Improvement over Baseline: +{(target_precision - base_rate_precision) * 100:.1f} percentage points")

Baseline Precision (Random/Always Predict 1): 54.2%
Target 'Good' Precision for Model: >= 75.0%
Required Improvement over Baseline: +20.8 percentage points


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis
One row = one unique content item.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


total_rows = len(df)
unique_content_ids = df['content_id'].nunique()

print(f"Total rows in dataset: {total_rows:,}")
print(f"Unique content IDs: {unique_content_ids:,}")

if total_rows == unique_content_ids:
    print("Grain Confirmed: 1 row = 1 unique content item.")
else:
    print("Warning: Duplicates exist.")


columns_to_show = ['content_id', 'client_id', 'content_type', 'trend_pct']
display(df[columns_to_show].head(3))

Total rows in dataset: 30,000
Unique content IDs: 30,000
Grain Confirmed: 1 row = 1 unique content item.


,content_id,client_id,content_type,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,-60.9


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Content decay exists in a highly non-linear, multi-dimensional feature space. A simple if statement is far too rigid. Some older content is evergreen and perfectly stable, while some brand-new content decays immediately after a short trend. Training complex tree based models allows us to figure out interactions across dozens of variables.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

features_to_check = ['content_age_days', 'ctr']
summary_stats = df.groupby('is_decaying')[features_to_check].agg(['median', 'mean', 'std'])

display(summary_stats)

print("The standard deviations are massive compared to the means.")
print("Because the distributions of decaying (1) and stable (0) pages overlap so heavily,")
print("a hardcoded if-statement threshold would generate massive amounts of false positives.")


content_age_days                            ctr                   
                      median        mean         std median     mean       std
is_decaying                                                                   
0                      287.0  279.855552  135.439012   0.04  0.73156  4.472882
1                      216.0  236.145836  126.957777   0.08  0.32408  1.689935

The standard deviations are massive compared to the means.
Because the distributions of decaying (1) and stable (0) pages overlap so heavily,
a hardcoded if-statement threshold would generate massive amounts of false positives.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.